# Thực Hành & Khảo Sát Cơ Chế Tokenization Trong Mô Hình BERT

**Học phần:** Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Nội dung:** Bài tập Tính toán 2 (Mục 10) & Cài đặt Cốt lõi Tokenization (Mục 12.1)  
**Mô hình khảo sát:** `bert-base-uncased` (WordPiece Tokenizer)  

---
Mục tiêu của notebook này là khảo sát chi tiết quy trình chuyển đổi từ văn bản thô sang các vector tensor chuẩn hóa mà mạng Transformer tiếp nhận, bao gồm:
1. Tách từ cấp độ từ con (Subword Tokenization)
2. Chèn các mã điều khiển đặc biệt (`[CLS]`, `[SEP]`, `[PAD]`)
3. Cơ chế hoạt động của mặt nạ chú ý (`attention_mask`)
4. Phân tích định lượng giữa Words, Tokens và Subwords

In [1]:
from transformers import AutoTokenizer

# Tải tokenizer của mô hình tiền huấn luyện bert-base-uncased
MODEL_CHECKPOINT = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
sample_sentence = "The cats are playing."

print(f"Đã tải thành công Tokenizer: {MODEL_CHECKPOINT}")
print(f"Kích thước bộ từ điển từ vựng (Vocab size): {tokenizer.vocab_size:,} mục từ")
print(f"Chuỗi câu đầu vào khảo sát: '{sample_sentence}'")

Đã tải thành công Tokenizer: bert-base-uncased
Kích thước bộ từ điển từ vựng (Vocab size): 30,522 mục từ
Chuỗi câu đầu vào khảo sát: 'The cats are playing.'


## 1 & 2. Tách Chuỗi Token (Token Sequence)

Khảo sát sự khác biệt giữa:
- **Raw tokens:** Các đơn vị từ/subword được chuẩn hóa chữ thường và tách dấu câu.
- **Full tokens:** Chuỗi token hoàn chỉnh được đóng gói với các token cấu trúc của BERT (`[CLS]` và `[SEP]`).

In [2]:
# 1. Tách từ con (subwords) bằng tokenizer.tokenize
subword_tokens = tokenizer.tokenize(sample_sentence)
print("1. Chuỗi token nội dung thô (WordPiece subwords):")
print("  ", subword_tokens)

# 2. Mã hóa hoàn chỉnh kèm token điều khiển
encoded_output = tokenizer(sample_sentence)
full_token_sequence = tokenizer.convert_ids_to_tokens(encoded_output["input_ids"])
print("\n2. Chuỗi token hoàn chỉnh nạp vào BERT Encoder (kèm Special Tokens):")
print("  ", full_token_sequence)

1. Chuỗi token nội dung thô (WordPiece subwords):
   ['the', 'cats', 'are', 'playing', '.']

2. Chuỗi token hoàn chỉnh nạp vào BERT Encoder (kèm Special Tokens):
   ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']


## 3, 4 & 5. Mã Định Danh (Token IDs), Token Đặc Biệt & Mặt Nạ Chú Ý (Attention Mask)

- `input_ids`: Chỉ số vị trí nguyên thủy của từng token trong bảng tra từ điển.
- `special_tokens`: Các token có chức năng cấu trúc điều khiển.
- `attention_mask`: Vector nhị phân giúp phân định giữa token thông tin hợp lệ ($1$) và khoảng đệm ($0$).

In [3]:
input_ids = encoded_output["input_ids"]
attention_mask = encoded_output["attention_mask"]

print("3. Danh sách Token IDs (input_ids):")
print("  ", input_ids)

print("\n4. Nhận diện các Special Tokens trong chuỗi:")
for idx, (tok, tid) in enumerate(zip(full_token_sequence, input_ids)):
    if tok == "[CLS]":
        print(f"   - Index {idx}: Token '{tok}' | ID = {tid} | Chức năng: Đại diện ngữ cảnh toàn câu đưa vào Linear Head")
    elif tok == "[SEP]":
        print(f"   - Index {idx}: Token '{tok}' | ID = {tid} | Chức năng: Ngăn cách/phân định ranh giới kết thúc mệnh đề")

print("\n5. Mặt nạ chú ý Attention Mask (chuỗi tiêu chuẩn):")
print("  ", attention_mask)

# Minh họa cơ chế đệm Padding theo batch
padded_data = tokenizer(sample_sentence, padding="max_length", max_length=10)
print("\n--- Trường hợp đệm chuỗi đồng bộ theo Batch (max_length = 10) ---")
print("Tokens sau khi đệm:", tokenizer.convert_ids_to_tokens(padded_data["input_ids"]))
print("Padded input_ids:   ", padded_data["input_ids"])
print("Padded attention_mask:", padded_data["attention_mask"])

3. Danh sách Token IDs (input_ids):
   [101, 1996, 8870, 2024, 2652, 1012, 102]

4. Nhận diện các Special Tokens trong chuỗi:
   - Index 0: Token '[CLS]' | ID = 101 | Chức năng: Đại diện ngữ cảnh toàn câu đưa vào Linear Head
   - Index 6: Token '[SEP]' | ID = 102 | Chức năng: Ngăn cách/phân định ranh giới kết thúc mệnh đề

5. Mặt nạ chú ý Attention Mask (chuỗi tiêu chuẩn):
   [1, 1, 1, 1, 1, 1, 1]

--- Trường hợp đệm chuỗi đồng bộ theo Batch (max_length = 10) ---
Tokens sau khi đệm: ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]']
Padded input_ids:    [101, 1996, 8870, 2024, 2652, 1012, 102, 0, 0, 0]
Padded attention_mask:[1, 1, 1, 1, 1, 1, 1, 0, 0, 0]


## 6. So Sánh Định Lượng: Words vs. Tokens vs. Subword Tokens

Khảo sát sự tương quan về mặt số lượng và bản chất ngôn ngữ học giữa ba cấp độ biểu diễn:

In [4]:
natural_words = [w.strip(".") for w in sample_sentence.split()]

print("+" + "-"*24 + "+" + "-"*10 + "+" + "-"*61 + "+")
print(f"| {'Cấp độ phân tích':<22} | {'Số lượng':<8} | {'Danh sách phần tử cụ thể':<59} |")
print("+" + "-"*24 + "+" + "-"*10 + "+" + "-"*61 + "+")
print(f"| {'Words (Từ tự nhiên)':<22} | {len(natural_words):^8} | {str(natural_words) + ' (và 1 dấu chấm \'.\')':<59} |")
print(f"| {'Tokens (Đầu vào BERT)':<22} | {len(full_token_sequence):^8} | {str(full_token_sequence):<59} |")
print(f"| {'Subwords (Mảnh từ vựng)':<22} | {len(subword_tokens):^8} | {str(subword_tokens):<59} |")
print("+" + "-"*24 + "+" + "-"*10 + "+" + "-"*61 + "+")

print("\nNhận xét về thuật toán WordPiece:")
print("- Các từ trong câu đều là những từ rất thông dụng trong tiếng Anh nên đã tồn tại sẵn")
print("  dưới dạng đơn vị nguyên khối trong từ điển 30,522 từ vựng của BERT.")
print("- Do đó, số lượng subword tokens nội dung trùng khớp với số từ ngữ nghĩa (4 từ + 1 dấu câu).")

+------------------------+----------+-------------------------------------------------------------+
| Cấp độ phân tích       | Số lượng | Danh sách phần tử cụ thể                                    |
+------------------------+----------+-------------------------------------------------------------+
| Words (Từ tự nhiên)    |    4     | ['The', 'cats', 'are', 'playing'] (và 1 dấu chấm '.')       |
| Tokens (Đầu vào BERT)  |    7     | ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']   |
| Subwords (Mảnh từ vựng)|    5     | ['the', 'cats', 'are', 'playing', '.']                      |
+------------------------+----------+-------------------------------------------------------------+

Nhận xét về thuật toán WordPiece:
- Các từ trong câu đều là những từ rất thông dụng trong tiếng Anh nên đã tồn tại sẵn
  dưới dạng đơn vị nguyên khối trong từ điển 30,522 từ vựng của BERT.
- Do đó, số lượng subword tokens nội dung trùng khớp với số từ ngữ nghĩa (4 từ + 1 dấu câu).


## 7. Khảo Sát Mở Rộng: Xử Lý Từ Hiếm / Ngoài Từ Điển (OOV & Subwords với `##`)

Để hiểu rõ cơ chế phân mảnh của thuật toán WordPiece khi đối mặt với từ ngữ hiếm gặp hoặc dị dạng chính tả, ta khảo sát câu biến thể: `"The catss are playingg."`

In [5]:
oov_sentence = "The catss are playingg."
oov_subwords = tokenizer.tokenize(oov_sentence)
print(f"Câu biến thể chứa từ hiếm: '{oov_sentence}'")
print("Các subwords sinh ra:", oov_subwords)

print("\nGiải thích tiền tố '##':")
print("- 'catss' được phân rã thành 'cats' + '##s' (ID tương ứng:", tokenizer.convert_tokens_to_ids(['cats', '##s']), ")")
print("- 'playingg' được phân rã thành 'playing' + '##g' (ID tương ứng:", tokenizer.convert_tokens_to_ids(['playing', '##g']), ")")
print("- Tiền tố '##' giúp BERT hiểu rằng đây là phần tiếp nối hình thái học của từ đứng trước,")
print("  giúp mô hình nắm bắt được gốc nghĩa mà không bao giờ gặp phải lỗi [UNK] (Unknown).")

Câu biến thể chứa từ hiếm: 'The catss are playingg.'
Các subwords sinh ra: ['the', 'cats', '##s', 'are', 'playing', '##g', '.']

Giải thích tiền tố '##':
- 'catss' được phân rã thành 'cats' + '##s' (ID tương ứng: 8870 + 2012)
- 'playingg' được phân rã thành 'playing' + '##g' (ID tương ứng: 2652 + 1043)
- Tiền tố '##' giúp BERT hiểu rằng đây là phần tiếp nối hình thái học của từ đứng trước,
  giúp mô hình nắm bắt được gốc nghĩa mà không bao giờ gặp phải lỗi [UNK] (Unknown).


## 8. Cài Đặt Cốt Lõi: Kiểm Tra `tokens.keys()` Theo Yêu Cầu Mục 12.1

Đoạn mã dưới đây thực thi đúng yêu cầu tại Mục 12.1 của đề bài, kiểm tra cấu trúc từ điển `tokens` và tensor PyTorch:

In [6]:
# Gọi tokenizer chuẩn hóa theo Mục 12.1
tokens = tokenizer(
    sample_sentence,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Các khóa trong tokens.keys():")
for k in tokens.keys():
    print(f"  -> Key: '{k:<20}' | Tensor shape: {tokens[k].shape} | Dtype: {tokens[k].dtype}")

print("\nChi tiết các tensor đầu ra:")
print("  input_ids:     ", tokens["input_ids"])
print("  token_type_ids:", tokens["token_type_ids"])
print("  attention_mask:", tokens["attention_mask"])

Các khóa trong tokens.keys():
  -> Key: 'input_ids'          | Tensor shape: torch.Size([1, 7]) | Dtype: torch.int64
  -> Key: 'token_type_ids'     | Tensor shape: torch.Size([1, 7]) | Dtype: torch.int64
  -> Key: 'attention_mask'     | Tensor shape: torch.Size([1, 7]) | Dtype: torch.int64

Chi tiết các tensor đầu ra:
  input_ids:      tensor([[ 101, 1996, 8870, 2024, 2652, 1012,  102]])
  token_type_ids: tensor([[0, 0, 0, 0, 0, 0, 0]])
  attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1]])
